In [1]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
torch.manual_seed(42)


In [2]:
ATTACK_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Spoofing\\combined_spoofing.csv"   # change per attack
BENIGN_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Benign\\combined_benign.csv"
MODEL_PATH   = "spoofing_bayesian.pth" 
SAMPLE_CAP  = 50000
RANDOM_SEED = 42

df_attack = pd.read_csv(ATTACK_FILE)
df_attack = df_attack.sample(n=min(len(df_attack), SAMPLE_CAP), random_state=RANDOM_SEED)

df_benign = pd.read_csv(BENIGN_FILE)
df_benign = df_benign.sample(n=len(df_attack), random_state=RANDOM_SEED)

# Combine and shuffle
df = pd.concat([df_benign, df_attack], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)

X = df.drop(columns=["Label"])
y = df["Label"]

# 60/20/20 split with stratification
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=RANDOM_SEED, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp)

print(f"Attack: {ATTACK_FILE} | Total rows: {len(df)}")
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Attack: C:\Users\johfr\Dropbox\Skule\Data science\Bachelor\Dataset\Spoofing\combined_spoofing.csv | Total rows: 100000
Train: 60000 | Val: 20000 | Test: 20000


In [3]:
from sklearn.preprocessing import KBinsDiscretizer

X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_val   = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# BN requires discrete features — bin continuous values into categories
discretizer = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
X_train_disc = pd.DataFrame(discretizer.fit_transform(X_train), columns=X_train.columns).astype(int)
X_val_disc   = pd.DataFrame(discretizer.transform(X_val),   columns=X_val.columns).astype(int)
X_test_disc  = pd.DataFrame(discretizer.transform(X_test),  columns=X_test.columns).astype(int)

# Combine with label for pgmpy
train_data = X_train_disc.copy()
train_data["label"] = y_train.values

C:\Users\johfr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\johfr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 2 are removed. Consider decreasing the number of bins.
  warnings.warn(
C:\Users\johfr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\preprocessing\

In [4]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

TOP_K = 20  # tune this — lower is faster, higher is more accurate

selector = SelectKBest(mutual_info_classif, k=TOP_K)
selector.fit(X_train_disc, y_train)

selected_cols = X_train_disc.columns[selector.get_support()].tolist()
print(f"Selected features: {selected_cols}")

X_train_disc = X_train_disc[selected_cols]
X_val_disc   = X_val_disc[selected_cols]
X_test_disc  = X_test_disc[selected_cols]

# Rebuild train_data with only selected features
train_data = X_train_disc.copy()
train_data["Label"] = y_train.values

Selected features: ['Dst Port', 'Flow Duration', 'Total Fwd Packet', 'Total Length of Fwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'Average Packet Size']


In [5]:
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BIC
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import BayesianEstimator

print("Learning structure...")
hc     = HillClimbSearch(train_data)
best_model_structure = hc.estimate(scoring_method=BIC(train_data), max_indegree=3)


print("Edges found:", best_model_structure.edges())

C:\Users\johfr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Dst Port': 'N', 'Flow Duration': 'N', 'Total Fwd Packet': 'N', 'Total Length of Fwd Packet': 'N', 'Fwd Packet Length Max': 'N', 'Fwd Packet Length Min': 'N', 'Fwd Packet Length Mean': 'N', 'Bwd Packet Length Mean': 'N', 'Flow Bytes/s': 'N', 'Flow Packets/s': 'N', 'Flow IAT Mean': 'N', 'Flow IAT Max': 'N', 'Flow IAT Min': 'N', 'Fwd Packets/s': 'N', 'Bwd Packets/s': 'N', 'Packet Length Max': 'N', 'Packet Length Mean': 'N', 'Packet Length Std': 'N', 'Packet Length Variance': 'N', 'Average Packet Size': 'N', 'Label': 'N'}
INFO:pgmpy: Datatype (N=n

Learning structure...


  0%|          | 54/1000000 [00:03<16:45:35, 16.57it/s]

Edges found: [('Dst Port', 'Bwd Packet Length Mean'), ('Dst Port', 'Fwd Packet Length Min'), ('Flow Duration', 'Flow IAT Max'), ('Flow Duration', 'Dst Port'), ('Flow Duration', 'Bwd Packets/s'), ('Flow Duration', 'Label'), ('Total Fwd Packet', 'Total Length of Fwd Packet'), ('Total Fwd Packet', 'Flow Duration'), ('Total Fwd Packet', 'Flow IAT Min'), ('Total Fwd Packet', 'Fwd Packet Length Mean'), ('Total Fwd Packet', 'Flow IAT Max'), ('Total Fwd Packet', 'Fwd Packets/s'), ('Total Fwd Packet', 'Packet Length Max'), ('Total Fwd Packet', 'Flow IAT Mean'), ('Total Fwd Packet', 'Fwd Packet Length Max'), ('Total Length of Fwd Packet', 'Fwd Packet Length Max'), ('Total Length of Fwd Packet', 'Dst Port'), ('Total Length of Fwd Packet', 'Average Packet Size'), ('Total Length of Fwd Packet', 'Flow Duration'), ('Fwd Packet Length Max', 'Packet Length Max'), ('Fwd Packet Length Max', 'Fwd Packet Length Min'), ('Fwd Packet Length Max', 'Flow IAT Mean'), ('Fwd Packet Length Min', 'Average Packet Siz

In [6]:
from pgmpy.models import DiscreteBayesianNetwork

bn_model = DiscreteBayesianNetwork(best_model_structure.edges())
bn_model.fit(train_data, estimator=BayesianEstimator, prior_type="BDeu")
print("Model fitted.")

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Dst Port': 'N', 'Flow Duration': 'N', 'Total Fwd Packet': 'N', 'Total Length of Fwd Packet': 'N', 'Fwd Packet Length Max': 'N', 'Fwd Packet Length Min': 'N', 'Fwd Packet Length Mean': 'N', 'Bwd Packet Length Mean': 'N', 'Flow Bytes/s': 'N', 'Flow Packets/s': 'N', 'Flow IAT Mean': 'N', 'Flow IAT Max': 'N', 'Flow IAT Min': 'N', 'Fwd Packets/s': 'N', 'Bwd Packets/s': 'N', 'Packet Length Max': 'N', 'Packet Length Mean': 'N', 'Packet Length Std': 'N', 'Packet Length Variance': 'N', 'Average Packet Size': 'N', 'Label': 'N'}


Model fitted.


In [7]:
from pgmpy.inference import VariableElimination

infer = VariableElimination(bn_model)

preds = []
for _, row in X_test_disc.iterrows():
    evidence = row.to_dict()
    result   = infer.map_query(variables=["Label"], evidence=evidence, show_progress=False)
    preds.append(result["Label"])

print(classification_report(y_test, preds, target_names=["Benign", "Attack"]))
print(confusion_matrix(y_test, preds))

              precision    recall  f1-score   support

      Benign       0.64      0.81      0.72     10000
      Attack       0.74      0.55      0.63     10000

    accuracy                           0.68     20000
   macro avg       0.69      0.68      0.67     20000
weighted avg       0.69      0.68      0.67     20000

[[8107 1893]
 [4509 5491]]
